In [0]:
from pyspark.sql import functions as F

ACCOUNT = "mynetflixproject"
CATALOG = "netflix_catalog"
SILVER  = f"{CATALOG}.net_schema"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def write_gold_table(df, table_name: str):
    clean_tbl = table_name.strip()
    gold_path = f"abfss://gold@{ACCOUNT}.dfs.core.windows.net/{clean_tbl}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(gold_path)
    )
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD}.{clean_tbl}
        USING DELTA
        LOCATION '{gold_path}'
    """)
    print(f"✅ Built & registered {GOLD}.{clean_tbl} ({df.count():,} rows)")

# Load Silver tables from netflix_catalog.net_schema
titles_df     = spark.table(f"{SILVER}.netflix_titles")
directors_df  = spark.table(f"{SILVER}.netflix_directors")
cast_df       = spark.table(f"{SILVER}.netflix_cast")
countries_df  = spark.table(f"{SILVER}.netflix_countries")
categories_df = spark.table(f"{SILVER}.netflix_category")

# ------------------------------------------------------------------------------
# 1. dim_titles (Descriptive attributes)
# ------------------------------------------------------------------------------
dim_titles = titles_df.select(
    "show_id",
    "title",
    "type",
    "rating",
    "release_year",
    "date_added",
    "description",
    "is_added_before_release"
)
write_gold_table(dim_titles, "dim_titles")

# ------------------------------------------------------------------------------
# 2. dim_date (Calendar table from min to max date_added)
# ------------------------------------------------------------------------------
date_bounds = titles_df.filter(F.col("date_added").isNotNull()).agg(
    F.min("date_added").alias("min_dt"),
    F.max("date_added").alias("max_dt")
).collect()[0]

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{date_bounds.min_dt}'), to_date('{date_bounds.max_dt}'), interval 1 day)) AS full_date")
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month_num", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("month_short", F.date_format("full_date", "MMM"))
    .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week_num", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.when(F.dayofweek("full_date").isin(1, 7), True).otherwise(False))
)
write_gold_table(dim_date, "dim_date")

# ------------------------------------------------------------------------------
# 3. fact_netflix_titles (Central fact table with pre-aggregated counts & lag metrics)
# ------------------------------------------------------------------------------
dir_counts  = directors_df.groupBy("show_id").agg(F.count("*").alias("director_count"))
cast_counts = cast_df.groupBy("show_id").agg(F.count("*").alias("cast_count"))
ctry_counts = countries_df.groupBy("show_id").agg(F.count("*").alias("country_count"))
cat_counts  = categories_df.groupBy("show_id").agg(F.count("*").alias("category_count"))

fact_netflix_titles = (
    titles_df
    .join(dir_counts,  on="show_id", how="left")
    .join(cast_counts, on="show_id", how="left")
    .join(ctry_counts, on="show_id", how="left")
    .join(cat_counts,  on="show_id", how="left")
    .select(
        F.col("show_id"),
        F.date_format("date_added", "yyyyMMdd").cast("int").alias("date_added_key"),
        F.col("release_year"),
        F.col("duration_minutes"),
        F.col("duration_seasons"),
        F.when(
            ~F.col("is_added_before_release") & F.col("date_added").isNotNull(),
            F.year("date_added") - F.col("release_year")
        ).alias("years_to_netflix"),
        F.coalesce(F.col("director_count"), F.lit(0)).alias("director_count"),
        F.coalesce(F.col("cast_count"),     F.lit(0)).alias("cast_count"),
        F.coalesce(F.col("country_count"),  F.lit(0)).alias("country_count"),
        F.coalesce(F.col("category_count"), F.lit(0)).alias("category_count"),
        F.current_timestamp().alias("_gold_processed_at")
    )
)
write_gold_table(fact_netflix_titles, "fact_netflix_titles")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def build_dim_and_bridge(silver_df, value_col: str, dim_name: str, bridge_name: str, key_col: str, clean_col_name: str):
    # Create deterministic surrogate key ordered alphabetically
    w = Window.orderBy(F.col(value_col))
    dim_df = (
        silver_df
        .select(F.col(value_col))
        .distinct()
        .withColumn(key_col, F.row_number().over(w))
        .select(F.col(key_col), F.col(value_col).alias(clean_col_name))
    )
    
    bridge_df = (
        silver_df.alias("s")
        .join(dim_df.alias("d"), F.col(f"s.{value_col}") == F.col(f"d.{clean_col_name}"), "inner")
        .select(F.col("s.show_id"), F.col(f"d.{key_col}"))
        .distinct()
    )
    
    write_gold_table(dim_df, dim_name)
    write_gold_table(bridge_df, bridge_name)

build_dim_and_bridge(directors_df,  "director",  "dim_directors",  "bridge_title_directors",  "director_key", "director_name")
build_dim_and_bridge(cast_df,       "cast",      "dim_cast",       "bridge_title_cast",       "cast_key",     "actor_name")
build_dim_and_bridge(countries_df,  "country",   "dim_countries",  "bridge_title_countries",  "country_key",  "country_name")
build_dim_and_bridge(categories_df, "listed_in", "dim_categories", "bridge_title_categories", "category_key", "category_name")


In [0]:


# ==============================================================================
# 4. FINAL GOLD STAR SCHEMA VALIDATION CHECK
# ==============================================================================
checks = [
    ("dim_titles", "show_id"),
    ("dim_date", "date_key"),
    ("fact_netflix_titles", "show_id"),
    ("dim_directors", "director_key"),
    ("bridge_title_directors", None),
    ("dim_cast", "cast_key"),
    ("bridge_title_cast", None),
    ("dim_countries", "country_key"),
    ("bridge_title_countries", None),
    ("dim_categories", "category_key"),
    ("bridge_title_categories", None),
]

rows = []
for tbl, pk in checks:
    df = spark.table(f"{GOLD}.{tbl}")
    total = df.count()
    uniq = df.select(pk).distinct().count() if (pk and pk in df.columns) else total
    assert total == uniq, f"❌ Duplicate primary keys found in {GOLD}.{tbl}!"
    rows.append((tbl, total, uniq))

print("\n🎉 ALL 11 GOLD TABLES BUILT, REGISTERED, AND VERIFIED!")
spark.createDataFrame(rows, ["table_name", "row_count", "unique_keys"]).display()
